# Huấn luyện Piper TTS với VietTTS-v1.1
Notebook này tải bộ dữ liệu VietTTS từ Hugging Face, chuyển metadata sang định dạng Piper, rồi chuẩn bị các bước preprocess, fine-tune và export model.

Nguồn dữ liệu: https://huggingface.co/datasets/ntt123/viet-tts-dataset/resolve/main/viet-tts.tar.gz

Ghi chú sử dụng:
- Chạy trên Linux, WSL hoặc Colab.
- Cần một checkpoint Piper tương thích nếu muốn fine-tune.
- Dataset sau khi giải nén sẽ có thư mục wav và file meta_data.tsv.

In [13]:
!pip install onnxscript

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/689.1 kB ? eta -:--:--
   ---------------------------------------- 689.1/689.1 kB 7.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   -------------------- ------------------- 6.3/12.4 MB 33.8 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.4 MB 27.0 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 27.1 MB/s  0:00:00
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 16.4/16.4 MB 93.9 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached typing_ex

## 1. Thiết lập đường dẫn
Thiết lập thư mục làm việc, nơi chứa dataset đã tải về, thư mục preprocess và thư mục xuất model.

In [14]:
from pathlib import Path

WORK_DIR = Path("train_test") / "viet_tts_work"
RAW_ARCHIVE = WORK_DIR / "viet-tts.tar.gz"
DATASET_DIR = WORK_DIR / "dataset"
TRAINING_DIR = WORK_DIR / "training"
EXPORT_DIR = WORK_DIR / "export"
PIPER_REPO = WORK_DIR / "piper1-gpl"

SPLIT_DIR = WORK_DIR / "split"
TRAIN_INPUT_DIR = SPLIT_DIR / "train"
TEST_INPUT_DIR = SPLIT_DIR / "test"
TEST_PREP_DIR = WORK_DIR / "test_preprocessed"

DATASET_URL = "https://huggingface.co/datasets/ntt123/viet-tts-dataset/resolve/main/viet-tts.tar.gz"
LANGUAGE = "vi"
SAMPLE_RATE = 22050
QUALITY = "medium"
BATCH_SIZE = 16
EPOCHS = 1000
BASE_CHECKPOINT = ""
TEST_RATIO = 0.1
TEST_SEED = 42

for folder in [
    WORK_DIR,
    DATASET_DIR,
    TRAINING_DIR,
    EXPORT_DIR,
    SPLIT_DIR,
    TRAIN_INPUT_DIR,
    TEST_INPUT_DIR,
    TEST_PREP_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORK_DIR.resolve())
print("Dataset directory:", DATASET_DIR.resolve())
print("Piper repo path:", PIPER_REPO.resolve())

Working directory: C:\Users\tiana\PRoject\vietnamese-text-analyzer\train_eval\train_test\viet_tts_work
Dataset directory: C:\Users\tiana\PRoject\vietnamese-text-analyzer\train_eval\train_test\viet_tts_work\dataset


## 2. Tải và giải nén dataset
Bộ dữ liệu VietTTS-v1.1 đi kèm file nén tar.gz. Ô này tải file về và giải nén vào thư mục dataset.

In [15]:
import tarfile
import urllib.request


def is_within_directory(directory: Path, target: Path) -> bool:
    directory = directory.resolve()
    target = target.resolve()
    return str(target).startswith(str(directory))


def safe_extract(tar: tarfile.TarFile, path: Path) -> None:
    for member in tar.getmembers():
        member_path = path / member.name
        if not is_within_directory(path, member_path):
            raise RuntimeError(f"Unsafe path in archive: {member.name}")
    tar.extractall(path)


if not RAW_ARCHIVE.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATASET_URL, RAW_ARCHIVE)
else:
    print("Dataset archive already exists:", RAW_ARCHIVE)

print("Extracting dataset...")
with tarfile.open(RAW_ARCHIVE, "r:gz") as tar:
    safe_extract(tar, DATASET_DIR)

print("Top-level files:")
for item in sorted(DATASET_DIR.iterdir()):
    print(" -", item.name)

Dataset archive already exists: train_test\viet_tts_work\viet-tts.tar.gz
Extracting dataset...
Top-level files:
 - collections.txt
 - meta_data.tsv
 - metadata.csv
 - wav


## 3. Chuyển meta_data.tsv sang metadata.csv cho Piper
Piper preprocess mong đợi metadata.csv với delimiter là dấu gạch đứng. Cột đầu là tên file audio, cột cuối là text. Nếu có cột speaker ở giữa, notebook sẽ giữ lại.

In [16]:
import csv

meta_tsv = DATASET_DIR / "meta_data.tsv"
metadata_csv = DATASET_DIR / "metadata.csv"

rows = []
with meta_tsv.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        cleaned = [col.strip() for col in row if col and col.strip()]
        if cleaned:
            rows.append(cleaned)

if rows:
    first_row_lower = {value.lower() for value in rows[0]}
    if first_row_lower & {"text", "transcript", "audio", "wav", "speaker", "file", "path"}:
        rows = rows[1:]

print("Preview rows:")
for row in rows[:3]:
    print(row)

normalized_rows = []
for row in rows:
    if len(row) == 2:
        filename, text = row
        normalized_rows.append([filename, text])
    elif len(row) >= 3:
        filename, speaker, text = row[0], row[1], row[-1]
        normalized_rows.append([filename, speaker, text])
    else:
        raise ValueError(f"Unexpected row format: {row}")

with metadata_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    writer.writerows(normalized_rows)

print("Wrote:", metadata_csv)
print("Example normalized row:", normalized_rows[0])

Preview rows:
['wav/000000.wav', 'Ai đây tức là một kẻ ăn mày vậy. Anh ta chưa kịp quay đi thì đã thấy mấy con chó vàng chạy xồng xộc ra cứ nhảy xổ vào chân anh.']
['wav/000001.wav', 'Anh mau cứu tôi! những lời nói run sợ một cách đáng lạ lùng kia, ngay bây giờ, ngồi viết những dòng này, tôi thấy như bên tai vẫn còn văng vẳng!...']
['wav/000002.wav', 'Anh đi đường cái quan đi ba bước rồi dừng lại, nhìn... Dưới ruộng chiêm, tiếng mấy người đàn bà cười khúc khích. Chợt khách bộ hành cũng hát:']
Wrote: train_test\viet_tts_work\dataset\metadata.csv
Example normalized row: ['wav/000000.wav', 'Ai đây tức là một kẻ ăn mày vậy. Anh ta chưa kịp quay đi thì đã thấy mấy con chó vàng chạy xồng xộc ra cứ nhảy xổ vào chân anh.']


## 4. Cài Piper training (theo OHF-Voice/piper1-gpl)
Phần này dùng đúng hướng dẫn trong docs/TRAINING.md: clone repo, cài `.[train]`, build monotonic alignment và build extension.

In [17]:
import os
import subprocess
import sys

if not PIPER_REPO.exists():
    subprocess.run([
        "git",
        "clone",
        "https://github.com/OHF-Voice/piper1-gpl.git",
        str(PIPER_REPO),
    ], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], cwd=str(PIPER_REPO), check=True)

print("Installed piper1-gpl training dependencies.")

Cloning into 'piper1-gpl'...
'source' is not recognized as an internal or external command,
operable program or batch file.
ERROR: '.[train]' is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).


In [18]:
import platform
import subprocess
import sys

if platform.system().lower() == "linux":
    subprocess.run(["bash", "./build_monotonic_align.sh"], cwd=str(PIPER_REPO), check=True)
else:
    print("Skip build_monotonic_align.sh on non-Linux environment.")

subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=str(PIPER_REPO), check=True)
print("Build extension completed.")

c:\Users\tiana\PRoject\vietnamese-text-analyzer\.venv\Scripts\python.exe: Error while finding module specification for 'piper.train.export_onnx' (ModuleNotFoundError: No module named 'piper')


In [19]:
import subprocess
import sys

train_cmd = [
    sys.executable,
    "-m",
    "piper.train",
    "fit",
    "--data.voice_name",
    "viettts",
    "--data.csv_path",
    str(metadata_csv),
    "--data.audio_dir",
    str(DATASET_DIR / "wav"),
    "--model.sample_rate",
    str(SAMPLE_RATE),
    "--data.espeak_voice",
    LANGUAGE,
    "--data.cache_dir",
    str(WORK_DIR / "cache"),
    "--data.config_path",
    str(TRAINING_DIR / "config.json"),
    "--data.batch_size",
    str(BATCH_SIZE),
    "--trainer.max_epochs",
    str(EPOCHS),
]

if BASE_CHECKPOINT:
    train_cmd.extend(["--ckpt_path", BASE_CHECKPOINT])

subprocess.run(train_cmd, cwd=str(PIPER_REPO), check=True)
print("Training command finished.")

CalledProcessError: Command '['c:\\Users\\tiana\\PRoject\\vietnamese-text-analyzer\\.venv\\Scripts\\python.exe', '-m', 'piper_train.preprocess', '--language', 'vi', '--input-dir', 'train_test\\viet_tts_work\\dataset', '--output-dir', 'train_test\\viet_tts_work\\training', '--dataset-format', 'ljspeech', '--single-speaker', '--sample-rate', '22050']' returned non-zero exit status 1.

## 5. Chia train/test cho đánh giá
Phần này chia `normalized_rows` thành train/test độc lập. Metadata của mỗi tập được lưu riêng để preprocess tách biệt.

In [ ]:
import random

if not normalized_rows:
    raise ValueError("No rows found to split train/test.")

rows_for_split = normalized_rows.copy()
random.Random(TEST_SEED).shuffle(rows_for_split)

test_size = max(1, int(len(rows_for_split) * TEST_RATIO))
if test_size >= len(rows_for_split):
    test_size = max(1, len(rows_for_split) - 1)

test_rows = rows_for_split[:test_size]
train_rows = rows_for_split[test_size:]

if not train_rows:
    raise ValueError("Train split is empty. Reduce TEST_RATIO.")


def to_absolute_audio_path(row):
    file_col = row[0]
    file_path = Path(file_col)
    if not file_path.is_absolute():
        file_path = (DATASET_DIR / file_col).resolve()

    if len(row) == 2:
        return [str(file_path), row[1]]
    return [str(file_path), row[1], row[2]]


train_rows_abs = [to_absolute_audio_path(row) for row in train_rows]
test_rows_abs = [to_absolute_audio_path(row) for row in test_rows]

train_metadata = TRAIN_INPUT_DIR / "metadata.csv"
test_metadata = TEST_INPUT_DIR / "metadata.csv"

with train_metadata.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    writer.writerows(train_rows_abs)

with test_metadata.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    writer.writerows(test_rows_abs)

print(f"Total rows: {len(rows_for_split)}")
print(f"Train rows: {len(train_rows_abs)} -> {train_metadata}")
print(f"Test rows: {len(test_rows_abs)} -> {test_metadata}")

## 6. Preprocess tách biệt train/test
Cell này cài Piper (nếu cần) và chạy `piper_train.preprocess` cho từng tập. Dùng `TRAINING_DIR` để train, và `TEST_PREP_DIR` cho đánh giá.

In [ ]:
import subprocess
import sys

if not PIPER_REPO.exists():
    subprocess.run([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/rhasspy/piper.git",
        str(PIPER_REPO),
    ], check=True)

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(PIPER_REPO / "src/python/requirements.txt"),
], check=True)

split_jobs = [
    ("train", TRAIN_INPUT_DIR, TRAINING_DIR),
    ("test", TEST_INPUT_DIR, TEST_PREP_DIR),
]

for split_name, input_dir, output_dir in split_jobs:
    output_dir.mkdir(parents=True, exist_ok=True)
    preprocess_cmd = [
        sys.executable,
        "-m",
        "piper_train.preprocess",
        "--language",
        LANGUAGE,
        "--input-dir",
        str(input_dir),
        "--output-dir",
        str(output_dir),
        "--dataset-format",
        "ljspeech",
        "--single-speaker",
        "--sample-rate",
        str(SAMPLE_RATE),
    ]
    subprocess.run(preprocess_cmd, cwd=str(PIPER_REPO / "src/python"), check=True)
    print(f"Preprocess completed for {split_name}: {output_dir}")

In [ ]:
train_jsonl = TRAINING_DIR / "dataset.jsonl"
test_jsonl = TEST_PREP_DIR / "dataset.jsonl"

if train_jsonl.exists() and test_jsonl.exists():
    train_count = sum(1 for _ in train_jsonl.open("r", encoding="utf-8"))
    test_count = sum(1 for _ in test_jsonl.open("r", encoding="utf-8"))
    print("Train samples after preprocess:", train_count)
    print("Test samples after preprocess:", test_count)
else:
    print("Chưa thấy dataset.jsonl cho train/test. Hãy chạy cell preprocess trước.")